In [ ]:
# Meesho CoD Trust Analysis
# Author: Vishal | IIT Madras BS Data Science
# Goal: Investigate root cause of Meesho's Cash on Delivery dominance
# Data: 133,300 self-scraped Google Play Store reviews

In [ ]:
# ── SECTION 1: SCRAPE REVIEWS ──────────────────────────────────────────────
# Scraping Meesho's Play Store reviews sorted newest first
# target_date sets how far back we go
# time.sleep(2) prevents Google from rate-limiting the scraper
 
from google_play_scraper import reviews, Sort
import pandas as pd
import time
import datetime

all_reviews = []
token = None
target_date = datetime.datetime(2024, 4, 1)

while True:
    batch, token = reviews(
        'com.meesho.supply',
        lang='en',
        country='in',
        sort=Sort.NEWEST,
        count=100,
        continuation_token=token
    )
    all_reviews.extend(batch)
    
    oldest = min(r['at'] for r in batch)
    print(f"Collected: {len(all_reviews)} | Oldest in batch: {oldest}")
    
    if oldest < target_date:
        break
    
    time.sleep(2)

df = pd.DataFrame(all_reviews)
df_filtered = df[df['score'] <= 2]
df_filtered.to_csv('meesho_reviews.csv', index=False)
print(f"Done. Total 1-2 star reviews saved: {len(df_filtered)}")

In [ ]:
# ── SECTION 2: BUILD AND SAVE DATAFRAME ──────────────────────────────────

df = pd.DataFrame(all_reviews)
df.to_csv('meesho_all_reviews_partial.csv', index=False)
print(f"Saved {len(df)} reviews")
print(f"Date range: {df['at'].min()} to {df['at'].max()}")

In [ ]:
# ── SECTION 3: FILTER NEGATIVE REVIEWS ────────────────────────────────────
# Keep only 1 and 2 star reviews — complaint signal lives here

df_filtered = df[df['score'] <= 2]
print(df_filtered['at'].min())
print(df_filtered['at'].max())
print(len(df_filtered))

In [ ]:
# ── SECTION 4: MONTHLY TREND ───────────────────────────────────────────────
# Identify complaint volume spikes month by month

df_filtered['month'] = df_filtered['at'].dt.to_period('M')
monthly_counts = df_filtered.groupby('month').size()
print(monthly_counts)

In [ ]:
# ── SECTION 5: KEYWORD ANALYSIS ───────────────────────────────────────────
# Map complaint categories across all negative reviews
# Financial keywords combined = 23.5% — second highest after delivery

keywords = ['payment', 'cod', 'refund', 'money', 'deliver', 
            'return', 'fraud', 'fake', 'support', 'cancel']

for keyword in keywords:
    count = df_filtered['content'].str.contains(keyword, case=False, na=False).sum()
    print(f"{keyword}: {count}")

In [ ]:
# ── SECTION 6: FRAUD + PAYMENT INTERSECTION ───────────────────────────────
# Isolate reviews where fraud complaint is specifically about money
# not product quality — direct evidence of payment trust failure

payment_fraud = df_filtered[
    df_filtered['content'].str.contains('fraud', case=False, na=False) &
    df_filtered['content'].str.contains('payment|refund|money|paise|deduct|debit', case=False, na=False)
]

print(f"Total fraud reviews: {df_filtered['content'].str.contains('fraud', case=False, na=False).sum()}")
print(f"Fraud + payment/refund/money: {len(payment_fraud)}")
print()
print("--- Sample reviews ---")
for review in payment_fraud['content'].head(30).tolist():
    print(review)
    print()